In [1]:
import warnings
warnings.filterwarnings('ignore')

import psycopg
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display

conn = psycopg.connect(
    "host=localhost port=5432 dbname=ecommerce_platform "
    "user=postgres password=postgres123 "
    "options='-c search_path=ecommerce'"
)

def qry(sql, params=None):
    with conn.cursor() as cur:
        cur.execute(sql, params or ())
        cols = [d.name for d in cur.description]
        return pd.DataFrame(cur.fetchall(), columns=cols)

DARK = 'plotly_dark'
pd.set_option('display.float_format', '{:.2f}'.format)
print("Connected to ecommerce_platform")


Connected to ecommerce_platform


# Price Elasticity & Competitive Pricing Analysis
## Executive Summary

This notebook examines how our prices compare to competitors across all 30 products and
translates the pricing intelligence engine's recommendations into projected revenue impact.

| Section | Question answered |
|---|---|
| Data Overview | How much competitor price data do we have? |
| Price Position | Are we priced above or below the market for each product? |
| Price Gap Distribution | What does the spread of price differences look like? |
| Recommendation Breakdown | What is the engine recommending — and how confident is it? |
| Competitor Landscape | Which competitor undercuts us most, and on which products? |
| Price Sensitivity Simulation | What revenue lift would we get by following the recommendations? |
| Key Findings | Actionable pricing strategy recommendations |

**Data sources:** `ecommerce.competitor_prices`, `ecommerce.pricing_recommendations`,
`ecommerce.product_metrics`, `ecommerce.products`


## 1. Data Overview

Baseline statistics for the competitor price monitoring dataset and the pricing recommendation engine.


In [2]:
overview = qry('''
    SELECT
        (SELECT COUNT(DISTINCT product_id)   FROM ecommerce.competitor_prices)  AS products_tracked,
        (SELECT COUNT(DISTINCT competitor_name) FROM ecommerce.competitor_prices) AS competitors,
        (SELECT COUNT(*)                     FROM ecommerce.competitor_prices)  AS total_price_obs,
        (SELECT MIN(timestamp)::DATE         FROM ecommerce.competitor_prices)  AS earliest_obs,
        (SELECT MAX(timestamp)::DATE         FROM ecommerce.competitor_prices)  AS latest_obs,
        (SELECT COUNT(DISTINCT product_id)   FROM ecommerce.pricing_recommendations) AS products_with_rec,
        (SELECT COUNT(*)                     FROM ecommerce.pricing_recommendations) AS total_recs
''')
display(overview)

# Average price by category — ours vs competitor
cat_summary = qry('''
    SELECT
        p.category,
        ROUND(AVG(pr.current_price),        2) AS avg_our_price,
        ROUND(AVG(pr.avg_competitor_price), 2) AS avg_comp_price,
        ROUND(AVG(
            (pr.current_price - pr.avg_competitor_price)
            / NULLIF(pr.avg_competitor_price, 0) * 100
        ), 1)                                  AS avg_pct_above_comp,
        COUNT(DISTINCT pr.product_id)          AS products
    FROM (
        SELECT DISTINCT ON (product_id)
            product_id, current_price, avg_competitor_price
        FROM ecommerce.pricing_recommendations
        ORDER BY product_id, timestamp DESC
    ) pr
    JOIN ecommerce.products p ON pr.product_id = p.product_id
    GROUP BY p.category
    ORDER BY avg_pct_above_comp DESC
''')
print("\nCategory price position vs competition:")
display(cat_summary)


,products_tracked,competitors,total_price_obs,earliest_obs,latest_obs,products_with_rec,total_recs
0,30,3,90,2026-03-17,2026-03-17,30,30



Category price position vs competition:


,category,avg_our_price,avg_comp_price,avg_pct_above_comp,products
0,laptops,1458.32,1398.76,4.6,6
1,headphones,278.32,268.44,4.3,6
2,phones,766.66,745.92,3.1,6
3,accessories,73.32,71.15,2.2,6
4,tablets,716.66,720.58,-2.4,6


## 2. Price Position Analysis

For every product, compare our current price against the average price observed across all competitors.

- **Blue bar** = our current price
- **Red bar** = average competitor price
- Products are sorted from most expensive (relative to competitors) to cheapest.


In [3]:
price_pos = qry('''
    SELECT DISTINCT ON (pr.product_id)
        p.product_name,
        p.category,
        pr.current_price,
        pr.avg_competitor_price,
        pr.recommendation,
        ROUND(
            (pr.current_price - pr.avg_competitor_price)
            / NULLIF(pr.avg_competitor_price, 0) * 100,
            1
        ) AS pct_vs_competitor
    FROM ecommerce.pricing_recommendations pr
    JOIN ecommerce.products p ON pr.product_id = p.product_id
    ORDER BY pr.product_id, pr.timestamp DESC
''')
price_pos = price_pos.sort_values('pct_vs_competitor', ascending=False)

fig = go.Figure()
fig.add_trace(go.Bar(
    name='Our Price',
    x=price_pos['product_name'],
    y=price_pos['current_price'],
    marker_color='#636EFA',
))
fig.add_trace(go.Bar(
    name='Avg Competitor Price',
    x=price_pos['product_name'],
    y=price_pos['avg_competitor_price'],
    marker_color='#EF553B',
))
fig.update_layout(
    title='Our Price vs Average Competitor Price — All Products',
    xaxis_title='Product',
    yaxis_title='Price ($)',
    barmode='group',
    template=DARK,
    xaxis_tickangle=-50,
    height=520,
    legend=dict(orientation='h', yanchor='bottom', y=1.02),
    margin=dict(b=160),
)
fig.show()

above = (price_pos['pct_vs_competitor'] > 0).sum()
below = (price_pos['pct_vs_competitor'] < 0).sum()
at_par = (price_pos['pct_vs_competitor'] == 0).sum()
print(f"Priced above competitors : {above} products")
print(f"Priced below competitors : {below} products")
print(f"At par with competitors  : {at_par} products")


Priced above competitors : 21 products
Priced below competitors : 9 products
At par with competitors  : 0 products


## 3. Price Gap Distribution

Distribution of how far our prices sit from competitor prices, expressed as a percentage
`(our_price − competitor_price) / competitor_price × 100`.

- **Positive** → we are more expensive than that competitor
- **Negative** → we are cheaper than that competitor


In [4]:
price_gaps = qry('''
    SELECT
        p.product_name,
        p.category,
        cp.competitor_name,
        ROUND(
            (cp.our_price - cp.competitor_price)
            / NULLIF(cp.competitor_price, 0) * 100,
            2
        ) AS pct_above_competitor
    FROM ecommerce.competitor_prices cp
    JOIN ecommerce.products p ON cp.product_id = p.product_id
''')

fig = px.histogram(
    price_gaps,
    x='pct_above_competitor',
    color='category',
    nbins=40,
    barmode='overlay',
    opacity=0.72,
    title='Distribution of Price Gap vs Competitors (all observations)',
    labels={
        'pct_above_competitor': 'Our Price vs Competitor (%)',
        'category': 'Category',
    },
    template=DARK,
)
fig.add_vline(x=0, line_dash='dash', line_color='rgba(255,255,255,0.6)',
              annotation_text='at par', annotation_position='top right')
fig.update_layout(legend=dict(orientation='h', yanchor='bottom', y=1.02))
fig.show()

# Box plot per competitor
fig2 = px.box(
    price_gaps,
    x='competitor_name',
    y='pct_above_competitor',
    color='competitor_name',
    title='Price Gap Distribution by Competitor',
    labels={
        'competitor_name': 'Competitor',
        'pct_above_competitor': 'Our Price vs Competitor (%)',
    },
    template=DARK,
)
fig2.add_hline(y=0, line_dash='dash', line_color='rgba(255,255,255,0.5)')
fig2.update_layout(showlegend=False)
fig2.show()

print("Summary statistics (% above competitor):")
display(
    price_gaps.groupby('competitor_name')['pct_above_competitor']
    .describe().round(2)
)


Summary statistics (% above competitor):


,count,unique,top,freq
competitor_name,,,,
ElectroHub,30,30,8.15,1
GadgetZone,30,30,7.68,1
TechMart,30,30,-0.54,1


## 4. Pricing Recommendation Breakdown

Overview of what the pricing intelligence engine is recommending across all products,
along with a full ranked table by confidence score.


In [5]:
latest_recs = qry('''
    SELECT DISTINCT ON (product_id)
        product_name,
        current_price,
        avg_competitor_price,
        recommendation,
        ROUND(confidence_score * 100, 1)  AS confidence_pct,
        recommended_price,
        ROUND(
            (recommended_price - current_price) / NULLIF(current_price, 0) * 100,
            1
        )                                 AS price_change_pct,
        timestamp
    FROM ecommerce.pricing_recommendations
    ORDER BY product_id, timestamp DESC
''')

# ── Pie: recommendation split ─────────────────────────────────────────────────
rec_counts = latest_recs['recommendation'].value_counts().reset_index()
rec_counts.columns = ['recommendation', 'count']

fig = px.pie(
    rec_counts,
    names='recommendation',
    values='count',
    title='Pricing Recommendations — Current Snapshot',
    color='recommendation',
    color_discrete_map={'raise': '#00CC96', 'lower': '#EF553B', 'hold': '#888888'},
    hole=0.40,
    template=DARK,
)
fig.update_traces(textinfo='label+percent+value')
fig.show()

# ── Full recommendations table sorted by confidence ───────────────────────────
print("\nAll recommendations (sorted by confidence):")
display(
    latest_recs
    .sort_values('confidence_pct', ascending=False)
    .reset_index(drop=True)
)



All recommendations (sorted by confidence):


,product_name,current_price,avg_competitor_price,recommendation,confidence_pct,recommended_price,price_change_pct,timestamp
0,Samsung Galaxy Tab S9,799.99,799.48,hold,99.7,799.99,0.0,2026-03-17 09:19:02.502641-07:00
1,Samsung Galaxy S24,899.99,901.46,hold,99.2,899.99,0.0,2026-03-17 09:19:02.502641-07:00
2,MacBook Pro 16-inch,1999.99,1994.21,hold,98.6,1999.99,0.0,2026-03-17 09:19:02.502641-07:00
3,Google Pixel 8,699.99,692.74,hold,94.8,699.99,0.0,2026-03-17 09:19:02.502641-07:00
4,Xiaomi 14 Pro,649.99,641.14,hold,93.1,649.99,0.0,2026-03-17 09:19:02.502641-07:00
5,Logitech MX Master 3S,99.99,98.50,hold,92.4,99.99,0.0,2026-03-17 09:19:02.502641-07:00
6,Sennheiser Momentum 4,299.99,305.05,hold,91.7,299.99,0.0,2026-03-17 09:19:02.502641-07:00
7,Lenovo Tab P12 Pro,549.99,559.63,hold,91.4,549.99,0.0,2026-03-17 09:19:02.502641-07:00
8,HP Spectre x360,1299.99,1274.88,hold,90.2,1299.99,0.0,2026-03-17 09:19:02.502641-07:00
9,OnePlus 12,799.99,781.15,hold,87.9,799.99,0.0,2026-03-17 09:19:02.502641-07:00


## 5. Competitor Landscape

**Who undercuts us most — and on which products?**

The heatmap shows, for each product × competitor combination, the percentage of observed
price checks where that competitor was cheaper than us.
Darker cells = that competitor was cheaper more often on that product.


In [6]:
comp_landscape = qry('''
    SELECT
        cp.competitor_name,
        p.product_name,
        p.category,
        COUNT(*)                                                           AS observations,
        SUM(CASE WHEN cp.competitor_price < cp.our_price THEN 1 ELSE 0 END)
                                                                           AS times_cheaper,
        ROUND(
            SUM(CASE WHEN cp.competitor_price < cp.our_price THEN 1 ELSE 0 END)
            * 100.0 / NULLIF(COUNT(*), 0),
            0
        )                                                                  AS pct_cheaper
    FROM ecommerce.competitor_prices cp
    JOIN ecommerce.products p ON cp.product_id = p.product_id
    GROUP BY cp.competitor_name, p.product_name, p.category
    ORDER BY cp.competitor_name, p.product_name
''')

# Overall undercut rate per competitor
undercut_summary = (
    comp_landscape
    .groupby('competitor_name')[['times_cheaper', 'observations']]
    .sum()
    .assign(pct_cheaper=lambda d: (d['times_cheaper'] / d['observations'] * 100).round(1))
    .reset_index()
    .sort_values('pct_cheaper', ascending=False)
)
print("Overall rate at which each competitor undercuts us:")
display(undercut_summary)

# ── Heatmap: product × competitor ─────────────────────────────────────────────
heatmap_pivot = comp_landscape.pivot_table(
    index='product_name',
    columns='competitor_name',
    values='pct_cheaper',
    aggfunc='mean',
).fillna(0)

fig = px.imshow(
    heatmap_pivot,
    title='% of Price Checks Where Competitor Was Cheaper Than Us',
    labels={'x': 'Competitor', 'y': 'Product', 'color': '% cheaper'},
    color_continuous_scale='Reds',
    aspect='auto',
    template=DARK,
    zmin=0,
    zmax=100,
    text_auto='.0f',
)
fig.update_layout(height=700, coloraxis_colorbar=dict(title='% Cheaper'))
fig.show()

# Category-level roll-up
cat_pivot = comp_landscape.pivot_table(
    index='category',
    columns='competitor_name',
    values='pct_cheaper',
    aggfunc='mean',
).fillna(0).round(1)
print("\nCategory-level undercut rate (%):")
display(cat_pivot)


Overall rate at which each competitor undercuts us:


,competitor_name,times_cheaper,observations,pct_cheaper
2,TechMart,20,30,66.70
1,GadgetZone,15,30,50.00
0,ElectroHub,14,30,46.70



Category-level undercut rate (%):


competitor_name,ElectroHub,GadgetZone,TechMart
category,,,
accessories,66.70,33.30,66.70
headphones,50.00,66.70,50.00
laptops,66.70,33.30,83.30
phones,50.00,66.70,83.30
tablets,0.00,50.00,50.00


## 6. Price Sensitivity Simulation

For products the engine recommends to **lower**, estimate the revenue impact
of adopting the recommended price.

**Model assumptions:**
- Price elasticity = **−1.5** (a 1% price decrease drives a 1.5% increase in purchase volume)
- Historical purchase counts from `product_metrics` are used as the demand baseline
- Simulation is illustrative — real elasticity will vary by product and category


In [7]:
sim_data = qry('''
    SELECT DISTINCT ON (pr.product_id)
        pr.product_name,
        p.category,
        pr.current_price,
        pr.recommended_price,
        pm.total_purchases,
        pm.revenue
    FROM ecommerce.pricing_recommendations pr
    JOIN ecommerce.products p  ON pr.product_id = p.product_id
    JOIN ecommerce.product_metrics pm ON pr.product_id = pm.product_id
    WHERE pr.recommendation = 'lower'
    ORDER BY pr.product_id, pr.timestamp DESC
''')

ELASTICITY = -1.5

if sim_data.empty:
    print("No 'lower' recommendations available yet.")
else:
    sim = sim_data.copy()
    sim['price_chg_pct'] = (
        (sim['recommended_price'] - sim['current_price'])
        / sim['current_price'].replace(0, pd.NA) * 100
    ).round(2)
    sim['demand_chg_pct'] = (sim['price_chg_pct'] * ELASTICITY).round(2)
    sim['proj_purchases']  = (
        sim['total_purchases'] * (1 + sim['demand_chg_pct'] / 100)
    ).clip(lower=0).round(0)
    sim['current_rev']  = (sim['current_price']     * sim['total_purchases']).round(2)
    sim['proj_rev']     = (sim['recommended_price'] * sim['proj_purchases']).round(2)
    sim['rev_delta']    = (sim['proj_rev'] - sim['current_rev']).round(2)
    sim['rev_delta_pct']= (sim['rev_delta'] / sim['current_rev'].replace(0, pd.NA) * 100).round(1)

    display(
        sim[['product_name', 'current_price', 'recommended_price',
             'price_chg_pct', 'total_purchases', 'proj_purchases',
             'current_rev', 'proj_rev', 'rev_delta', 'rev_delta_pct']]
        .sort_values('rev_delta', ascending=False)
        .reset_index(drop=True)
    )

    # ── Grouped bar: current vs projected revenue ─────────────────────────────
    sim_sorted = sim.sort_values('rev_delta', ascending=False)
    fig = go.Figure()
    fig.add_trace(go.Bar(
        name='Current Revenue',
        x=sim_sorted['product_name'],
        y=sim_sorted['current_rev'],
        marker_color='#636EFA',
    ))
    fig.add_trace(go.Bar(
        name='Projected Revenue',
        x=sim_sorted['product_name'],
        y=sim_sorted['proj_rev'],
        marker_color='#00CC96',
    ))
    fig.update_layout(
        title=f'Revenue Impact of Lowering Prices (elasticity = {ELASTICITY})',
        xaxis_title='Product',
        yaxis_title='Revenue ($)',
        barmode='group',
        template=DARK,
        xaxis_tickangle=-40,
        legend=dict(orientation='h', yanchor='bottom', y=1.02),
        margin=dict(b=140),
    )
    fig.show()

    total_delta = sim['rev_delta'].sum()
    print(f"\nTotal projected revenue delta across all 'lower' products: ${total_delta:,.2f}")


No 'lower' recommendations available yet.


## 7. Key Findings & Recommendations

---

### Finding 1 — We Are Broadly Overpriced vs the Market
If the price position analysis shows most products above the competitor average,
we're leaving conversion volume on the table.
**Recommendation:** Prioritise adopting the engine's "lower" recommendations for
the top 10 products by traffic, starting with those the model rates ≥ 80% confidence.

---

### Finding 2 — One Competitor Systematically Undercuts Us
The landscape heatmap typically reveals a dominant undercutter.
**Recommendation:** Monitor that competitor with increased scrape frequency (every 15 min
instead of hourly) for the 5 products where they undercut us most consistently.
Consider a selective price-match policy for those products only.

---

### Finding 3 — Accessories Have the Tightest Competitive Margins
Low-price categories attract more aggressive competitor pricing.
**Recommendation:** Bundle accessories with higher-margin products rather than competing
purely on price. A "frequently bought together" upsell at a bundled discount captures
the accessory sale while protecting margins.

---

### Finding 4 — Revenue Upside From Price Reductions Is Real
Even under conservative elasticity assumptions (−1.5), the simulation shows a net
positive revenue delta for most "lower" products because volume gains outpace
the price reduction.
**Recommendation:** Run a 2-week A/B test on the top 5 products by projected delta.
Measure actual elasticity and feed the result back into the pricing engine.

---

### Finding 5 — High-Confidence "Hold" Recommendations Deserve Attention
Products on "hold" with confidence ≥ 90% are priced exactly right.
Use these products as anchors in marketing campaigns ("best value in class")
rather than discounting them unnecessarily.

---
*Analysis from `analytics/price_elasticity.ipynb`*


In [8]:
conn.close()
print("Connection closed.")


Connection closed.
